# 10 — Monitoring and alert thresholds

This notebook simulates a feature distribution shift and applies a simple alert policy. In a real deployment, send alerts to a monitored channel (email, Slack, PagerDuty) and include ownership and runbook links.

In [1]:
import json, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def psi(reference, current, bins=10):
    edges=np.unique(np.quantile(reference, np.linspace(0,1,bins+1)))
    if len(edges)<3: return 0.0
    edges[0],edges[-1]=-np.inf,np.inf
    a=np.histogram(reference,bins=edges)[0]/len(reference); b=np.histogram(current,bins=edges)[0]/len(current)
    eps=1e-6
    return float(np.sum((b-a)*np.log((b+eps)/(a+eps))))
feature=X.columns[0]
ref=X_train[feature].to_numpy()
simulated_current= X_test[feature].to_numpy()*1.35
score=psi(ref, simulated_current)
alert={'check':'feature_drift','feature':feature,'psi':score,'threshold':0.2,'status':'ALERT' if score>=0.2 else 'OK'}
print(alert)
if alert['status']=='ALERT': print('Action: inspect upstream data, compare feature distributions, and evaluate labeled quality before retraining.')

{'check': 'feature_drift', 'feature': 'mean radius', 'psi': 3.4748343770241257, 'threshold': 0.2, 'status': 'ALERT'}
Action: inspect upstream data, compare feature distributions, and evaluate labeled quality before retraining.
